# Caderno Colab: ccRCC / TCGA-KIRC + Genomic Intelligence

**Versão de revisão:** R938 · 21/10/2026 · notebook prospectivo, ainda não executado.

> **Estado:** foi feita consulta metadata-only ao GDC; nenhum arquivo de conteúdo baixado e nenhum resultado TCGA observado. Este caderno não executa análise, R, `GDCdownload`, instala pacotes nem monta o Google Drive.

A abertura e o relatório final devem conservar essa distinção. Após autorização explícita do operador, cada execução Colab monta o Drive e cria um diretório de execução versionado `runs/<RUN_ID UTC>`, sem sobrescrever execuções anteriores; manifesto, dataset, logs e resultados pertencem a essa pasta. Testes do guard de manifesto são sintéticos e não validam o pipeline científico.

## 1. Pergunta e estimando prognóstico

A pergunta principal é se o modelo clínico-metabólico melhora a discriminação prognóstica em relação ao modelo clínico-base. O estimando primário é a **diferença da AUC cumulative/dynamic em 5 anos (60 meses), com IC de 95%**, calculada uma única vez no holdout somente se os critérios de estimabilidade, seguimento e censura forem satisfeitos. Não se presume superioridade.

MUI tumor-normal é análise secundária descritiva; assinaturas imunes são exploratórias. Genes, transformações, modelo, horizonte e comparador precisam estar congelados antes da avaliação. O desenho R934 requer validação cruzada aninhada no discovery, validação interna fechada e nenhuma seleção motivada por resultado favorável.

## 2. Proveniência GDC e estado dos gates (21/10/2026)

A consulta anterior foi somente de metadados: 614 file IDs únicos, 605 selecionados antes da exclusão autorizada, 2.566.647.024 bytes. O teto operacional autorizado é **6 GiB (6 × 1024³ bytes)**. O inventário deve ser regenerado antes de ser usado: a versão atual do manifesto ainda não registra a exclusão de workflow autorizada nem o release comum recalculado.

O arquivo `2b0106c5-7e32-423b-957f-06c9f2e7751b` foi autorizado para exclusão da coorte primária por `workflow_version` divergente. Seu `data_release` 36.0–46.0 se sobrepõe ao intervalo principal 32.0–46.0; igualdade textual de intervalos não é requisito. O GDC `/files/versions` fornece `latest_release` por versão; a elegibilidade depende da interseção dessas listas. A exclusão não se baseia no intervalo textual de release.

| Gate | Estado atual |
|---|---|
| P0 — protocolo | Condicional: parecer recebido requer correções e aceite metodológico; GREEN documental não valida a ciência |
| P1 — manifesto | Pendente: regenerar com exclusão autorizada, releases comuns e sidecar |
| P2 — dados | Pendente: nenhum conteúdo baixado; checksum do conteúdo não verificado |
| P3 — pipeline | Pendente: nenhuma execução; testes de fuga/nested CV do pipeline ainda necessários |
| P4 — análise | Pendente: nenhum resultado TCGA observado |

Não iniciar transferência nem análise enquanto o manifesto não passar pelo guard abaixo e os gates metodológicos humanos não forem fechados.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, re, shutil
import pandas as pd

# O operador autorizou persistência versionada no Drive via Colab.
MONTAR_DRIVE = True
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Execute este caderno em um runtime Colab; não use caminhos locais como substituto.') from exc
if MONTAR_DRIVE:
    drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/ccRCC_Genomic_Intelligence')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S.%fZ')
RUN_DIR = PROJECT_ROOT / 'runs' / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)  # cada execução é imutável; nunca sobrescreve outra
INPUT_ROOT = PROJECT_ROOT / 'inputs'
RUN_INPUT_DIR = RUN_DIR / 'inputs'
DATA_DIR = RUN_DIR / 'data'
TABLES_DIR = RUN_DIR / 'tables'
FIGURES_DIR = RUN_DIR / 'figures'
MODELS_DIR = RUN_DIR / 'models'
LOGS_DIR = RUN_DIR / 'logs'
for path in (INPUT_ROOT, RUN_INPUT_DIR, DATA_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, LOGS_DIR):
    path.mkdir(parents=True, exist_ok=True)
MAX_CONTENT_BYTES = 6 * 1024**3
AUTHORIZED_OUTLIER = '2b0106c5-7e32-423b-957f-06c9f2e7751b'
EXECUTAR_GI_REST = False  # nenhuma chamada REST do Genomic Intelligence nesta execução
P0_METHOD_ACCEPTED = False  # aceite metodológico humano permanece pendente
P1_HUMAN_REVIEW_ACCEPTED = False  # a aprovação do manifesto exige revisão independente
MANIFEST_SUMMARY = None
MANIFEST_SOURCE = INPUT_ROOT / 'gdc-manifest.json'
MANIFEST_SOURCE_SIDECAR = INPUT_ROOT / 'gdc-manifest.json.sha256'
MANIFEST_PATH = RUN_INPUT_DIR / 'gdc-manifest.json'
MANIFEST_SIDECAR_PATH = RUN_INPUT_DIR / 'gdc-manifest.json.sha256'
RUN_METADATA = {'run_id': RUN_ID, 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'output_root': str(RUN_DIR), 'drive_mounted': True, 'file_content_downloaded': False, 'analysis_executed': False}
(RUN_DIR / 'run_metadata.json').write_text(json.dumps(RUN_METADATA, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
def record_stage(stage, status, details=None):
    event = {'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'run_id': RUN_ID, 'stage': stage, 'status': status, 'details': details or {}}
    with (LOGS_DIR / 'stage_events.jsonl').open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(event, ensure_ascii=False, sort_keys=True) + '\n')
def validate_colab_execution_gate(operation, p0_method_accepted, p1_human_review_accepted, manifest_summary=None):
    if operation in {'metadata_only', 'synthetic_test', 'static_pilot'}:
        return True
    if operation not in {'download', 'analysis'}:
        raise ValueError('operation_not_allowlisted')
    if p0_method_accepted is not True:
        raise ValueError('P0_human_methodological_acceptance_required')
    if p1_human_review_accepted is not True:
        raise ValueError('P1_human_manifest_review_required')
    if not isinstance(manifest_summary, dict) or manifest_summary.get('status') != 'metadata_manifest_ready' or manifest_summary.get('download_eligible') is not True:
        raise ValueError('validated_download_eligible_manifest_required')
    if manifest_summary.get('manifest_sha256_verified') is not True:
        raise ValueError('manifest_sidecar_sha256_not_verified')
    if manifest_summary.get('file_content_downloaded') is not False or manifest_summary.get('analysis_executed') is not False:
        raise ValueError('manifest_not_metadata_only')
    return True
record_stage('colab_runtime_and_drive_mount', 'initialized', {'drive_mounted': MONTAR_DRIVE})
print('Run UTC versionado no Drive:', RUN_DIR, '| sem download/analysis nesta etapa.')

## 3. Piloto Genomic Intelligence — linha exploratória separada

Os números abaixo foram retornados por chamadas anteriores ao plugin a partir de sequências Ensembl GRCh38. Classificações de promotor, cromatina, splicing e TPM previstos são resultados *in silico*; não são medições de RNA-seq TCGA, diferença tumor-normal, medida por paciente ou validação prognóstica, e não entram no modelo prognóstico primário. Não somar TPM previsto a contagens TCGA. Uma nova chamada REST exigiria credencial do usuário e ativação explícita; o padrão deste notebook permanece desligado.

In [ ]:
PILOT = [
    {'gene': 'SLC2A1', 'ensembl_id': 'ENSG00000117394', 'sequence_bp': 37552,
     'promoter_windows': 38, 'predicted_promoter_windows': 13, 'promoter_max_score': 0.9860970378,
     'splice_donors': 11, 'splice_acceptors': 13, 'chromatin_windows': 188,
     'chromatin_annotations': 17531, 'predicted_TPM_reference': 39.9449},
    {'gene': 'SDHB', 'ensembl_id': 'ENSG00000117118', 'sequence_bp': 39547,
     'promoter_windows': 40, 'predicted_promoter_windows': 7, 'promoter_max_score': 0.8814655542,
     'splice_donors': None, 'splice_acceptors': None, 'chromatin_windows': 198,
     'chromatin_annotations': 4321, 'predicted_TPM_reference': 126.9445}
]
pilot = pd.DataFrame(PILOT)
pilot_path = TABLES_DIR / 'GI_pilot_observed_predictions.tsv'
pilot.to_csv(pilot_path, sep='\t', index=False)
record_stage('gi_pilot_static_predictions', 'written', {'path': str(pilot_path.relative_to(RUN_DIR)), 'sha256': hashlib.sha256(pilot_path.read_bytes()).hexdigest(), 'source': 'previous_plugin_output'})
display(pilot)

## 4. Gate local do manifesto metadata-only

Coloque o manifesto metadata-only atualizado e seu sidecar SHA-256 em `MyDrive/ccRCC_Genomic_Intelligence/inputs/`. O notebook copia ambos, sem modificar a origem, para a pasta `RUN_DIR/inputs/` de timestamp UTC e registra hashes/evento. O guard confere status, limite, file IDs, acesso open/released, MD5 informado pelo GDC, workflow, releases comuns e a exclusão autorizada. Ele **não** baixa conteúdo nem comprova o MD5 contra bytes transferidos. Enquanto P0/P1 não estiverem aprovados, a saída é BLOQUEADO e nenhuma transferência começa.

In [ ]:
def validate_gdc_manifest(manifest, sidecar_matches):
    problems = []
    if not isinstance(manifest, dict):
        raise ValueError('Manifesto BLOQUEADO: invalid_json_object')
    if sidecar_matches is not True:
        problems.append('sha256_sidecar_mismatch')
    if manifest.get('status') != 'metadata_manifest_ready':
        problems.append('manifest_not_ready')
    if manifest.get('download_eligible') is not True:
        problems.append('download_not_eligible')
    if manifest.get('max_content_bytes') != MAX_CONTENT_BYTES:
        problems.append('content_limit_not_6gib')
    if manifest.get('cohort_query_sent') is not True:
        problems.append('cohort_query_not_attested')
    if manifest.get('file_content_downloaded') is not False or manifest.get('controlled_accessed') is not False or manifest.get('analysis_executed') is not False:
        problems.append('metadata_only_flags_invalid')
    files_value = manifest.get('files')
    files = files_value if isinstance(files_value, list) else []
    if not isinstance(files_value, list) or any(not isinstance(row, dict) for row in files):
        problems.append('file_records_malformed')
    selected = [row for row in files if isinstance(row, dict) and row.get('selection_status') == 'selected_for_manifest']
    ids = [row.get('file_id') for row in selected]
    if not selected or any(not isinstance(x, str) or not x for x in ids) or len(ids) != len(set(ids)):
        problems.append('selected_file_ids_invalid_or_duplicate')
    if type(manifest.get('selected_file_count')) is not int or manifest.get('selected_file_count') != len(selected):
        problems.append('selected_file_count_mismatch')
    sizes_valid = all(type(row.get('file_size')) is int and row['file_size'] > 0 for row in selected)
    selected_bytes = manifest.get('selected_bytes')
    if not sizes_valid or type(selected_bytes) is not int or sum(row.get('file_size', 0) for row in selected) != selected_bytes:
        problems.append('selected_byte_sum_invalid')
    elif selected_bytes > MAX_CONTENT_BYTES:
        problems.append('selected_content_over_6gib')
    if any(row.get('access') != 'open' or row.get('state') != 'released' for row in selected):
        problems.append('selected_file_not_open_released')
    if any(not isinstance(row.get('md5sum'), str) or not re.fullmatch(r'[0-9a-f]{32}', row['md5sum']) for row in selected):
        problems.append('selected_md5_metadata_invalid')
    workflow_values = [row.get('workflow_version') for row in selected]
    workflow_versions = sorted(set(workflow_values)) if all(isinstance(x, str) and x for x in workflow_values) else []
    if len(workflow_versions) != 1 or workflow_versions != manifest.get('workflow_versions'):
        problems.append('workflow_version_not_homogeneous')
    common = manifest.get('common_data_releases')
    chosen = manifest.get('selected_common_data_release')
    if not isinstance(common, list) or not common or chosen not in common:
        problems.append('common_release_missing_or_unselected')
    else:
        release_sets = [set(row.get('available_releases', [])) for row in selected if isinstance(row.get('available_releases'), list)]
        if len(release_sets) != len(selected) or not release_sets:
            problems.append('selected_release_membership_missing')
        else:
            actual_common = set.intersection(*release_sets)
            if set(common) != actual_common or any(chosen not in releases for releases in release_sets):
                problems.append('common_release_intersection_mismatch')
            elif chosen != max(common, key=lambda value: tuple(int(part) for part in value.split('.'))):
                problems.append('selected_release_not_latest_common')
    excluded_rows = [row for row in files if isinstance(row, dict) and row.get('file_id') == AUTHORIZED_OUTLIER]
    excluded_records_value = manifest.get('authorized_exclusions')
    excluded_records = excluded_records_value if isinstance(excluded_records_value, list) else []
    if len(excluded_rows) != 1 or excluded_rows[0].get('selection_status') != 'excluded_workflow_version_incompatible':
        problems.append('authorized_workflow_outlier_not_recorded_as_excluded')
    elif excluded_rows[0].get('workflow_version') != '5a02c12930eb23436322a7494047b907b97570c5' or excluded_rows[0].get('workflow_version') in workflow_versions:
        problems.append('authorized_exclusion_workflow_mismatch_not_confirmed')
    if AUTHORIZED_OUTLIER in ids:
        problems.append('authorized_outlier_counted_as_primary')
    valid_exclusion = any(isinstance(record, dict) and record.get('file_id') == AUTHORIZED_OUTLIER and record.get('decision_basis') == 'workflow_version' and isinstance(record.get('rationale'), str) and record.get('rationale') for record in excluded_records)
    if not valid_exclusion:
        problems.append('authorized_exclusion_rationale_missing')
    if len([record for record in excluded_records if isinstance(record, dict) and record.get('file_id') == AUTHORIZED_OUTLIER]) != 1:
        problems.append('authorized_exclusion_record_count_invalid')
    if not any(isinstance(record, dict) and record.get('file_id') == AUTHORIZED_OUTLIER and record.get('authorized_on') == '2026-10-21' and record.get('workflow_version') == '5a02c12930eb23436322a7494047b907b97570c5' and isinstance(record.get('rationale'), str) and 'não é motivo de exclusão' in record['rationale'].casefold() for record in excluded_records):
        problems.append('authorized_exclusion_basis_or_rationale_mismatch')
    if problems:
        raise ValueError('Manifesto BLOQUEADO: ' + ', '.join(problems))
    return {
        'selected_file_count': len(selected),
        'selected_bytes': selected_bytes,
        'common_data_releases': common,
        'selected_common_data_release': chosen,
        'authorized_outlier_basis': 'workflow_version',
        'manifest_sha256_verified': True,
        'file_content_downloaded': False,
    }

def load_and_validate_gdc_manifest(manifest_path, sidecar_path):
    manifest_bytes = Path(manifest_path).read_bytes()
    expected = Path(sidecar_path).read_text(encoding='ascii').strip().split()[0]
    actual = hashlib.sha256(manifest_bytes).hexdigest()
    sidecar_matches = bool(re.fullmatch(r'[0-9a-f]{64}', expected)) and expected == actual
    manifest = json.loads(manifest_bytes.decode('utf-8'))
    return validate_gdc_manifest(manifest, sidecar_matches)

if MANIFEST_SOURCE.is_file() and MANIFEST_SOURCE_SIDECAR.is_file():
    shutil.copy2(MANIFEST_SOURCE, MANIFEST_PATH)
    shutil.copy2(MANIFEST_SOURCE_SIDECAR, MANIFEST_SIDECAR_PATH)
    try:
        manifest_summary = load_and_validate_gdc_manifest(MANIFEST_PATH, MANIFEST_SIDECAR_PATH)
        record_stage('metadata_manifest_preflight', 'passed', manifest_summary)
        print('Manifesto aprovado pelo guard:', manifest_summary)
    except (OSError, ValueError, KeyError, TypeError, json.JSONDecodeError) as exc:
        record_stage('metadata_manifest_preflight', 'blocked', {'reason': str(exc)})
        print('P1 BLOQUEADO — não baixar nem iniciar análise:', str(exc))
    else:
        try:
            validate_colab_execution_gate('analysis', P0_METHOD_ACCEPTED, P1_HUMAN_REVIEW_ACCEPTED, manifest_summary)
            record_stage('science_execution_gate', 'passed', {'operation': 'analysis'})
        except ValueError as exc:
            record_stage('science_execution_gate', 'blocked', {'reason': str(exc)})
            print('P0/P1 BLOQUEADOS — não iniciar análise:', str(exc))
else:
    record_stage('metadata_manifest_preflight', 'blocked', {'reason': 'manifest_and_sha256_sidecar_missing_from_drive_inputs'})
    print('P1 BLOQUEADO — coloque manifesto e sidecar atualizados na pasta Drive/inputs; não iniciar download.')

In [ ]:
def write_run_artifact_index():
    artifacts = []
    for base, dirs, names in os.walk(RUN_DIR):
        dirs[:] = sorted(d for d in dirs if d not in {'.ipynb_checkpoints', '__pycache__'})
        for name in sorted(names):
            path = Path(base) / name
            if path.name in {'artifact_index.json', 'artifact_index.json.sha256'} or not path.is_file():
                continue
            artifacts.append({'path': str(path.relative_to(RUN_DIR)), 'bytes': path.stat().st_size, 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
    index = {'run_id': RUN_ID, 'generated_at_utc': datetime.now(timezone.utc).isoformat(), 'artifact_count': len(artifacts), 'artifacts': artifacts, 'file_content_downloaded': False, 'analysis_executed': False}
    index_path = LOGS_DIR / 'artifact_index.json'
    index_bytes = (json.dumps(index, ensure_ascii=False, sort_keys=True, indent=2) + '\n').encode('utf-8')
    index_path.write_bytes(index_bytes)
    digest = hashlib.sha256(index_bytes).hexdigest()
    index_path.with_suffix('.json.sha256').write_text(digest + '\n', encoding='ascii')
    return {'path': str(index_path.relative_to(RUN_DIR)), 'sha256': digest, 'artifact_count': len(artifacts)}

print('Índice versionado do run:', write_run_artifact_index())

## 5. Pipeline TCGA e riscos do script anexado

Este notebook não incorpora R em Base64, não instala Bioconductor e não executa `TCGA_KIRC_SDD_TDD.R`. A inspeção estática do script anexado constatou que `GDCquery` não fixa `access=open`/`state=released` antes de `GDCdownload`, que falta o gate de manifesto/6 GiB, que uma chamada `cv.glmnet` no treino não demonstra CV aninhada outer/inner e que `OS_days >= 30` requer justificativa. Esse script não é autorizado como pipeline de aquisição/análise nesta versão. Uma futura implementação terá spec/teste próprios; o holdout permanecerá fechado.

## 6. Saída, transparência e diretrizes

Os testes do notebook e do guard usam dados sintéticos; não são resultados de pacientes nem evidência de validade clínica. O relatório precisa manter P0–P4 como pendentes/condicionais até cada gate ser demonstrado. Se não houver manifesto atualizado ou release comum, a saída correta é BLOQUEADO.

TRIPOD+AI (2024) e REMARK são guias de relato; não certificam validade, baixo risco de viés, validação externa ou utilidade clínica. O relatório prognóstico deve apresentar a diferença de AUC em 5 anos com IC 95%, denominadores, censura, missingness, calibração e limitações, sem impor resultado favorável.

Referências: Collins et al. (2024), TRIPOD+AI, DOI 10.1136/bmj-2023-078378; McShane et al. (2005), REMARK, DOI 10.1093/jnci/dji237; documentação oficial GDC Search and Retrieval e API `/files/versions`.